# Packet P-046 — Untapped Features Audit

**Decision question:** The dataset has ~30 unused columns including solvent ratios (DMF/DMSO, 92% fill), LLE composition embeddings (99.9% fill), thermal exposure (91% fill), layer thicknesses (ETL 45%, backcontact 85%), and PCE (100% fill). Do any of these improve LOGO cross-family generalization or random CV tau-b?

**Fastest falsifier:** Add all high-fill unused features (>80% fill) to the model. If LOGO tau-b stays <0.02 and random CV doesn't improve, these features are dead weight.

**Success:** LOGO tau-b ≥ 0.05 OR random CV tau-b ≥ 0.32 (meaningful lift over 0.289 baseline).
**Kill:** LOGO tau-b < 0.02 AND random CV < 0.30.

In [1]:
"""Cell 1 — Test untapped features: kitchen sink, then ablation."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import LeaveOneGroupOut, KFold
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')

# === Define feature sets ===
ORIGINAL_16 = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]

# New features grouped by category
SOLVENT_FEATURES = ['DMF', 'DMSO', 'other_solvent', 'DMF_DMSO_ratio']  # 92% fill
LLE_FEATURES = ['LLE_1', 'LLE_2', 'LLE_3', 'LLE_4']  # 99.9% fill
PROCESS_FEATURES = ['Perovskite_annealing_thermal_exposure']  # 91% fill
LAYER_FEATURES = ['Backcontact_thickness_list', 'ETL_thickness', 'HTL_thickness_list']  # 27-85% fill
OTHER_COMP = ['others_A', 'others_B', 'others_X']  # 99%+ fill
PCE_FEATURE = ['JV_default_PCE']  # 100% fill — NOTE: could be leakage-adjacent

# Kitchen sink: all high-fill features (>25% fill)
KITCHEN_SINK = ORIGINAL_16 + SOLVENT_FEATURES + LLE_FEATURES + PROCESS_FEATURES + LAYER_FEATURES + OTHER_COMP

# Kitchen sink + PCE (separate test since PCE is target-adjacent)
KITCHEN_SINK_PCE = KITCHEN_SINK + PCE_FEATURE

TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

y = np.log1p(df[TARGET])

def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)
groups = df["family"]

# === Test each feature group incrementally ===
feature_configs = {
    'Original 16': ORIGINAL_16,
    '+ Solvents (4)': ORIGINAL_16 + SOLVENT_FEATURES,
    '+ LLE embed (4)': ORIGINAL_16 + LLE_FEATURES,
    '+ Process (1)': ORIGINAL_16 + PROCESS_FEATURES,
    '+ Layers (3)': ORIGINAL_16 + LAYER_FEATURES,
    '+ Other comp (3)': ORIGINAL_16 + OTHER_COMP,
    'Kitchen sink (31)': KITCHEN_SINK,
    'Kitchen + PCE (32)': KITCHEN_SINK_PCE,
}

# === LOGO CV ===
print(f"{'=' * 75}")
print("LOGO CV — FEATURE GROUP ABLATION")
print(f"{'=' * 75}")

logo = LeaveOneGroupOut()
results = []

for config_name, feats in feature_configs.items():
    X = df[feats].fillna(0)
    
    all_y_true, all_y_pred = [], []
    family_taus = {}
    
    for train_idx, test_idx in logo.split(X, y, groups):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        held_out = groups.iloc[test_idx].iloc[0]
        
        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        tau, _ = kendalltau(y_te, preds)
        family_taus[held_out] = tau
        all_y_true.extend(y_te.tolist())
        all_y_pred.extend(preds.tolist())
    
    overall_tau, _ = kendalltau(all_y_true, all_y_pred)
    mean_fam_tau = np.mean(list(family_taus.values()))
    
    results.append({
        'config': config_name,
        'n_features': len(feats),
        'logo_overall_tau': overall_tau,
        'logo_mean_fam_tau': mean_fam_tau,
        'family_taus': family_taus,
    })
    
    print(f"\n{config_name} ({len(feats)} features):")
    print(f"  Overall LOGO tau-b: {overall_tau:.4f}  |  Mean family: {mean_fam_tau:.4f}")
    for fam in sorted(family_taus.keys()):
        print(f"    {fam:<20} {family_taus[fam]:+.4f}")

# === Random CV for top configs ===
print(f"\n{'=' * 75}")
print("RANDOM 5-FOLD CV — KEY CONFIGS")
print(f"{'=' * 75}")

N_REPEATS = 4
random_cv_results = {}

for config_name, feats in feature_configs.items():
    X = df[feats].fillna(0)
    all_taus = []
    
    for rep in range(N_REPEATS):
        kf = KFold(n_splits=5, shuffle=True, random_state=rep)
        for tr_idx, te_idx in kf.split(X):
            model = ExtraTreesRegressor(**{**ET_PARAMS, 'random_state': rep})
            model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            preds = model.predict(X.iloc[te_idx])
            tau, _ = kendalltau(y.iloc[te_idx], preds)
            all_taus.append(tau)
    
    mean_tau = np.mean(all_taus)
    random_cv_results[config_name] = mean_tau
    delta = mean_tau - random_cv_results.get('Original 16', mean_tau)
    print(f"  {config_name:<25} tau-b: {mean_tau:.4f}  delta: {delta:+.4f}")

# === Feature importance for kitchen sink ===
print(f"\n{'=' * 75}")
print("FEATURE IMPORTANCE — KITCHEN SINK MODEL")
print(f"{'=' * 75}")

X_ks = df[KITCHEN_SINK].fillna(0)
model_ks = ExtraTreesRegressor(**ET_PARAMS)
model_ks.fit(X_ks, y)
imp = pd.Series(model_ks.feature_importances_, index=KITCHEN_SINK).sort_values(ascending=False)

new_feats = set(KITCHEN_SINK) - set(ORIGINAL_16)
for feat, val in imp.items():
    marker = " ** NEW" if feat in new_feats else ""
    print(f"  {feat:<50} {val:.4f}{marker}")

LOGO CV — FEATURE GROUP ABLATION



Original 16 (16 features):
  Overall LOGO tau-b: 0.0180  |  Mean family: 0.0049
    FA-Cs                -0.0488
    MA-FA mixed          -0.0724
    Other                +0.0566
    Pure FA              +0.1420
    Pure MA              +0.0783
    Triple cation        -0.1264



+ Solvents (4) (20 features):
  Overall LOGO tau-b: 0.0350  |  Mean family: 0.0046
    FA-Cs                -0.1677
    MA-FA mixed          -0.0446
    Other                +0.0434
    Pure FA              +0.1371
    Pure MA              +0.0873
    Triple cation        -0.0282



+ LLE embed (4) (20 features):
  Overall LOGO tau-b: -0.0005  |  Mean family: 0.0358
    FA-Cs                +0.1720
    MA-FA mixed          -0.1185
    Other                +0.1166
    Pure FA              +0.0944
    Pure MA              +0.0604
    Triple cation        -0.1102



+ Process (1) (17 features):
  Overall LOGO tau-b: 0.0124  |  Mean family: 0.0019
    FA-Cs                -0.0425
    MA-FA mixed          -0.1237
    Other                +0.0761
    Pure FA              +0.1453
    Pure MA              +0.0732
    Triple cation        -0.1168


...

  + Layers (3)              tau-b: 0.3182  delta: +0.0222


  + Other comp (3)          tau-b: 0.2976  delta: +0.0016


  Kitchen sink (31)         tau-b: 0.3355  delta: +0.0395


  Kitchen + PCE (32)        tau-b: 0.3343  delta: +0.0383

FEATURE IMPORTANCE — KITCHEN SINK MODEL


  Cell_area_measured                                 0.0797
  JV_default_Jsc                                     0.0696
  JV_default_Voc                                     0.0671
  JV_default_FF                                      0.0602
  first_Prvskt_thermal_annealing_time                0.0588
  Backcontact_thickness_list                         0.0564 ** NEW
  Perovskite_annealing_thermal_exposure              0.0553 ** NEW
  Perovskite_thickness                               0.0531
  first_Prvskt_annealing_temperature                 0.0498
  ETL_thickness                                      0.0494 ** NEW
  Perovskite_band_gap                                0.0396
  HTL_thickness_list                                 0.0390 ** NEW
  DMF_DMSO_ratio                                     0.0356 ** NEW
  DMF                                                0.0337 ** NEW
  DMSO                                               0.0295 ** NEW
  LLE_2                                            

In [2]:
"""Cell 2 — Evaluate and save."""

orig_logo = results[0]['logo_overall_tau']  # Original 16
orig_random = random_cv_results['Original 16']

best_logo = max(r['logo_overall_tau'] for r in results)
best_logo_name = [r['config'] for r in results if r['logo_overall_tau'] == best_logo][0]
best_random = max(random_cv_results.values())
best_random_name = [k for k, v in random_cv_results.items() if v == best_random][0]

logo_pass = best_logo >= 0.05
random_pass = best_random >= 0.32

if logo_pass or random_pass:
    status = "Confirmed"
elif best_logo < 0.02 and best_random < 0.30:
    status = "Negative"
else:
    status = "Promising"

# Save
save_rows = []
for r in results:
    row = {
        'config': r['config'],
        'n_features': r['n_features'],
        'logo_overall_tau': r['logo_overall_tau'],
        'logo_mean_fam_tau': r['logo_mean_fam_tau'],
        'random_cv_tau': random_cv_results.get(r['config'], float('nan')),
    }
    for fam, tau in r['family_taus'].items():
        row[f'logo_{fam}'] = tau
    save_rows.append(row)

pd.DataFrame(save_rows).to_csv('Packet_P046_Untapped_Features.csv', index=False)
print(f"Saved: Packet_P046_Untapped_Features.csv")

print(f"\n{'=' * 60}")
print(f"P-046 STATUS: {status}")
print(f"{'=' * 60}")
print(f"Original:  LOGO={orig_logo:.4f}  Random CV={orig_random:.4f}")
print(f"Best LOGO: {best_logo:.4f} ({best_logo_name})")
print(f"Best Random CV: {best_random:.4f} ({best_random_name})")
print(f"LOGO pass (≥0.05): {logo_pass}")
print(f"Random CV pass (≥0.32): {random_pass}")

if status == "Confirmed":
    print("\nUntapped features provide meaningful lift!")
elif status == "Negative":
    print("\nUntapped features are dead weight — no improvement on either metric.")
else:
    print("\nSome improvement but below breakthrough thresholds.")
    print("May be worth cherry-picking the best individual feature groups.")

Saved: Packet_P046_Untapped_Features.csv

P-046 STATUS: Confirmed
Original:  LOGO=0.0180  Random CV=0.2960
Best LOGO: 0.0350 (+ Solvents (4))
Best Random CV: 0.3355 (Kitchen sink (31))
LOGO pass (≥0.05): False
Random CV pass (≥0.32): True

Untapped features provide meaningful lift!
